In [0]:
CATALOG = "databricks_project_tsv-752"
SCHEMA  = "healthcare"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
print(f"✅ Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
volumes = {
    "raw"         : "Landing zone — upload your source CSV files here",
    "bronze"      : "Delta tables written by the ingestion layer",
    "silver"      : "Enriched / transformed Delta tables",
    "gold"        : "Aggregated KPI views and summary tables",
    "checkpoints" : "Streaming checkpoint metadata (Auto Loader / Structured Streaming)",
    "streaming"   : "Landing folder Auto Loader monitors for new arriving CSVs",
}

In [0]:
for vol_name, description in volumes.items():
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{vol_name}`
    """)
    print(f"✅ Volume created: {CATALOG}.{SCHEMA}.{vol_name}  —  {description}")

In [0]:
print("=" * 72)
print("📂  UPLOAD YOUR FILES — EXACT PATHS")
print("=" * 72)

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

files = {
    "patient_visits.csv"        : "Main patient visits dataset  (200 rows)",
    "patient_visits_batch2.csv" : "Second batch for Auto Loader (30 rows)",
    "patients.csv"              : "Patient demographics          (30 rows)",
    "billing.csv"               : "Billing records               (200 rows)",
}

print(f"\n  Upload destination folder:\n  {BASE}\n")
print(f"  {'File':<35} {'Description'}")
print(f"  {'-'*35} {'-'*35}")
for fname, desc in files.items():
    print(f"  {fname:<35} {desc}")
    print(f"    → {BASE}/{fname}")
    print()

In [0]:
# Raw CSV locations (after you upload the files)
RAW_VOL           = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
VISITS_CSV        = f"{RAW_VOL}/patient_visits.csv"
VISITS_BATCH2_CSV = f"{RAW_VOL}/patient_visits_batch2.csv"
PATIENTS_CSV      = f"{RAW_VOL}/patients.csv"
BILLING_CSV       = f"{RAW_VOL}/billing.csv"

# Delta table storage locations (managed by Unity Catalog)
BRONZE_VOL        = f"/Volumes/{CATALOG}/{SCHEMA}/bronze"
SILVER_VOL        = f"/Volumes/{CATALOG}/{SCHEMA}/silver"
GOLD_VOL          = f"/Volumes/{CATALOG}/{SCHEMA}/gold"

# Streaming paths
CHECKPOINT_VOL    = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
STREAMING_VOL     = f"/Volumes/{CATALOG}/{SCHEMA}/streaming"

print("=" * 72)
print("📌  PIPELINE PATH CONSTANTS (copy into downstream notebooks)")
print("=" * 72)
print(f"""
CATALOG           = "{CATALOG}"
SCHEMA            = "{SCHEMA}"

RAW_VOL           = "{RAW_VOL}"
VISITS_CSV        = "{VISITS_CSV}"
VISITS_BATCH2_CSV = "{VISITS_BATCH2_CSV}"
PATIENTS_CSV      = "{PATIENTS_CSV}"
BILLING_CSV       = "{BILLING_CSV}"

BRONZE_VOL        = "{BRONZE_VOL}"
SILVER_VOL        = "{SILVER_VOL}"
GOLD_VOL          = "{GOLD_VOL}"
CHECKPOINT_VOL    = "{CHECKPOINT_VOL}"
STREAMING_VOL     = "{STREAMING_VOL}"
""")

In [0]:
print("=" * 72)
print("🔍  VOLUME VERIFICATION")
print("=" * 72)

result = spark.sql(f"SHOW VOLUMES IN `{CATALOG}`.`{SCHEMA}`")
display(result)


In [0]:
print("=" * 72)
print("✅  FILE UPLOAD VERIFICATION  (run after uploading CSVs)")
print("=" * 72)

import os

expected_files = [VISITS_CSV, VISITS_BATCH2_CSV, PATIENTS_CSV, BILLING_CSV]
all_ok = True

for fpath in expected_files:
    fname = fpath.split("/")[-1]
    try:
        # dbutils.fs works with /Volumes paths on Databricks Runtime 13.3+
        info = dbutils.fs.ls(fpath)
        size_kb = info[0].size / 1024
        print(f"  ✅  {fname:<35}  {size_kb:>8.1f} KB   found at {fpath}")
    except Exception:
        print(f"  ❌  {fname:<35}  NOT FOUND — please upload to {RAW_VOL}/")
        all_ok = False

print()
if all_ok:
    print("🎉  All files present — you can now run the Bronze ingestion notebook.")
else:
    print("⚠️   Some files are missing — upload them before running the pipeline.")